# Champaign, Urbana, Savoy Housing Market Analysis

This notebook walks through a hands-on market analysis of single-family homes sold in Champaign, Urbana, and Savoy. We use it as a worked example for the consulting scenario described in the [case overview](./case-overview.md): we are advising _Kingfisher Residential Partners_ on where and how to deploy a \$5–10M residential investment.

Our analysis follows five questions, in order:

1. **Market overview**: how do the three cities compare on price and inventory?
2. **Pricing drivers**: what property features explain the variation in price?
3. **Market liquidity**: can this dataset tell us how quickly properties sell? (A data check.)
4. **Temporal trends**: are prices rising, falling, or flat?
5. **Geographic insights**: where in the metro is the best value?

The notebook closes with a recap that turns the findings into recommendations an investor could act on.


## ⚙️ Setup

We lean on a small, focused stack: `pandas` for tabular work, `numpy` for numerics, and `plotly.express` for interactive charts. Plotly is well-suited to a real-estate notebook because hovering over a point lets us see the underlying property details, which is handy for sanity-checking individual listings.


In [1]:
import pandas as pd
import numpy as np
import plotly
import plotly.express as px
import plotly.io as pio

pio.templates.default = "simple_white"

print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Plotly version: {plotly.__version__}")

Pandas version: 3.1.1
Numpy version: 2.5.4
Plotly version: 7.1.0


In [2]:
# Set pandas display options to show more columns
pd.set_option("display.max_columns", 50)

## 📥 Load the Dataset

The data is published as a Parquet file on GitHub. Parquet is the format of choice for real-estate panel data because it is columnar (so reading just `price` and `living_area` is fast), it preserves the original dtypes (datetimes stay datetimes), and it is dramatically smaller than CSV.


In [3]:
df = pd.read_parquet(
    "https://media.githubusercontent.com/media/bdi593/datasets/main/zillow-properties/zillow_properties_champaign_urbana_savoy.parquet"
)
df.head(3)

,property_id,date_posted,days_on_zillow,price,description,beds,baths,year_built,latitude,longitude,lot_size,living_area,address_street,address_city,address_state,address_zipcode,reso_has_garage,reso_has_attached_garage,reso_garage_parking_capacity,reso_parking_capacity,reso_attic,reso_stories,reso_basement_yn,reso_has_cooling,reso_has_heating,reso_is_new_construction,reso_elementary_school,reso_middle_or_junior_school,reso_high_school,reso_tax_annual_amount
0,3232534,2022-05-24,1053,165000,This 3 bedroom ranch in an established neighbo...,3,1.0,1958,40.093950,-88.26074,10660.0,1436.0,1804 McDonald Dr,Champaign,IL,61821,True,1.0,1.0,1,NaN,1.0,1.0,1.0,1.0,0.0,Unit 4 Of Choice,Champaign/Middle Call Unit 4 351,Central High School,3270.0
1,3197892,2022-05-27,1028,900000,A TRULY SPECTACULAR HOME IN LINCOLNSHIRE FIELD...,4,6.0,1993,40.095110,-88.30598,32234.0,5430.0,1809 Cobblefield Ct,Champaign,IL,61822,True,1.0,3.0,3,NaN,2.0,1.0,1.0,1.0,0.0,Unit 4 Of Choice,Champaign/Middle Call Unit 4 351,Centennial High School,18782.0
2,3238411,2022-06-15,755,245000,"Wow! 711 W Green is a 5 bedroom, 2 full bath ...",5,2.0,1903,40.110363,-88.21728,6098.0,2046.0,711 W Green St,Urbana,IL,61801,False,0.0,NaN,2,NaN,2.0,1.0,1.0,1.0,0.0,Leal Elementary School,Urbana Middle School,Urbana High School,9413.0


Take a quick look at the schema. Note the mix of property attributes (`beds`, `baths`, `living_area`, `year_built`), location columns (`latitude`, `longitude`, `address_city`, `address_zipcode`), market-activity fields (`date_posted`, `days_on_zillow`), and school information (`reso_high_school`). This breadth is what makes the dataset useful for the five business questions.


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3737 entries, 0 to 3736
Data columns (total 30 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   property_id                   3737 non-null   int64         
 1   date_posted                   3299 non-null   datetime64[us]
 2   days_on_zillow                3737 non-null   int64         
 3   price                         3737 non-null   int64         
 4   description                   3683 non-null   str           
 5   beds                          3737 non-null   int64         
 6   baths                         3736 non-null   float64       
 7   year_built                    3737 non-null   int64         
 8   latitude                      3716 non-null   float64       
 9   longitude                     3716 non-null   float64       
 10  lot_size                      2998 non-null   float64       
 11  living_area                   3725 non-nu

### Data Cleaning

A quick profile of the columns we rely on shows three problems:

- One property has a `price` of 0, which cannot be a real sale price.
- Two properties have a `living_area` of 0. Dividing by zero later would make their price per square foot infinite, and an infinite value turns any average it touches into infinity.
- 438 properties have no `date_posted`. They are still useful for price comparisons, but they will drop out of the time-trend section.

We apply two simple rules: drop rows with a price of 0, and treat a living area of 0 as missing.


In [5]:
print(f"Rows with price == 0:        {(df['price'] == 0).sum()}")
print(f"Rows with living_area == 0:  {(df['living_area'] == 0).sum()}")
print(f"Rows with no date_posted:    {df['date_posted'].isna().sum()}")

df = df[df["price"] > 0].copy()
df["living_area"] = df["living_area"].replace(0, np.nan)

print(f"\nRows after cleaning: {len(df):,}")
print(f"Date range of date_posted: {df['date_posted'].min():%Y-%m-%d} to {df['date_posted'].max():%Y-%m-%d}")

Rows with price == 0:        1
Rows with living_area == 0:  2
Rows with no date_posted:    438

Rows after cleaning: 3,736
Date range of date_posted: 2022-05-24 to 2026-03-23


## 🏙️ 1. Market Overview

### Price Distribution by City

A box plot is the standard first look at price across geographies. The box shows the middle 50% of prices (P25 → P75), the line inside is the median, and the dots are individual listings the chart treats as outliers.


In [6]:
fig = px.box(
    df,
    y="price",
    x="address_city",
    title="Price Distribution by City",
    template="simple_white",
)
fig.show()

:::{tip} Why median, not mean?
Real-estate prices are right-skewed: a single \$2M custom home will pull the _mean_ sharply higher without being representative of the typical buyer's experience. The _median_ is what every CMA, every Zillow market report, and every appraiser uses for this reason.
:::

### Price per Square Foot

Raw price tells us about a property; **price per square foot** lets us compare properties of different sizes on the same yard-stick. It is the single most-used unit in real estate analytics.


In [7]:
df["price_per_sqft"] = df["price"] / df["living_area"]
df[["price", "living_area", "price_per_sqft"]].head(3)

,price,living_area,price_per_sqft
0,165000,1436.0,114.902507
1,900000,5430.0,165.745856
2,245000,2046.0,119.745846


In [8]:
fig = px.box(
    df,
    x="price_per_sqft",
    y="address_city",
    title="Price per Sqft by City",
    template="simple_white",
)
fig.show()

Champaign and Urbana overlap heavily: their typical price per square foot is very close. Savoy sits higher on this measure too, so Savoy homes are not only bigger (and newer), they also cost more per square foot. The table in the next section puts numbers on this.

### Median Price by City

A bar chart of median prices makes the city comparison crisp. For an investor, this is the headline of the market overview slide.


In [9]:
df_city_summary = (
    df.groupby("address_city")
    .agg(
        count=("property_id", "count"),
        median_price=("price", "median"),
        median_price_per_sqft=("price_per_sqft", "median"),
        median_living_area=("living_area", "median"),
        median_year_built=("year_built", "median"),
    )
    .reset_index()
)
display(df_city_summary.round(0))

fig1 = px.bar(
    df_city_summary,
    x="address_city",
    y="median_price",
    text_auto=".2s",
    title="Median Sale Price by City",
    labels={"median_price": "Median Price ($)", "address_city": "City"},
    template="simple_white",
    color="address_city",
)
fig1.show()

,address_city,count,median_price,median_price_per_sqft,median_living_area,median_year_built
0,Champaign,2325,240000.0,150.0,1644.0,1968.0
1,Savoy,254,405665.0,188.0,2164.0,2010.0
2,Urbana,1157,230000.0,145.0,1554.0,1966.0


## 💰 2. Pricing Drivers

With the high-level picture in mind, we can ask: _which property features actually drive price?_ We start with a correlation matrix on the obvious numeric candidates.


In [10]:
num_cols = [
    "price",
    "beds",
    "baths",
    "living_area",
    "year_built",
    "reso_tax_annual_amount",
]
corr = df[num_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    title="Price Drivers: Correlation Matrix",
    color_continuous_scale="RdBu_r",
    template="simple_white",
)
fig.show()

:::{caution} Correlation ≠ causation
The matrix above tells us which features _move together_ with price. It does not tell us that adding a bathroom causes a price jump: bigger houses tend to have more bathrooms _and_ sell for more, so the correlation captures both effects.
:::

### Living Area vs Price

Living area is almost always the single strongest predictor of price. The OLS trend line on each city makes the slope (_roughly the implied price per square foot_) directly visible.


In [11]:
# Price vs Living Area
fig = px.scatter(
    df,
    x="living_area",
    y="price",
    color="address_city",
    trendline="ols",
    title="Relationship: Living Area vs Price",
    labels={"living_area": "Living Area (sqft)", "price": "Price ($)"},
    hover_data=["address_street"],
    template="simple_white",
)
fig.update_traces(marker=dict(size=3))
fig.update_yaxes(range=[0, 1500000])
fig.show()

If we strip out the city colouring and the trend line, we can see the raw cloud of points. The fan-out at higher square-footage is classic: large luxury homes have very wide pricing because location and finish matter much more at the high end.


In [12]:
fig = px.scatter(
    df,
    x="living_area",
    y="price",
    opacity=0.4,
    title="Price vs Living Area",
    template="simple_white",
)
fig.update_yaxes(range=[0, 1000000])
fig.show()

### Bedrooms vs Price

Bedroom count rises with house size, so the chart below mixes the effect of an extra bedroom with the effect of extra square footage. Read it as "what bigger homes sell for", not as the value of adding a bedroom.


In [13]:
fig = px.box(
    df,
    x="beds",
    y="price",
    title="Price by Number of Bedrooms",
    template="simple_white",
)
fig.show()

### The Garage Premium

The chart below compares the prices of homes with and without a garage.


In [14]:
fig = px.box(
    df, x="price", y="reso_has_garage", title="Garage vs Price", template="simple_white"
)
fig.update_xaxes(range=[0, 2000000])
fig.show()

In [15]:
df.groupby("reso_has_garage").agg(
    homes=("property_id", "count"),
    median_price=("price", "median"),
    median_living_area=("living_area", "median"),
    median_year_built=("year_built", "median"),
)

,homes,median_price,median_living_area,median_year_built
reso_has_garage,,,,
False,718,150000.0,1300.5,1958.0
True,3018,267700.0,1757.0,1976.0


Homes with a garage sell for much more at the median, but they are also larger and newer. As with the correlation matrix, this gap is not the value of adding a garage. Estimating that would need a model that holds size, age, and location fixed.

## ⏱️ 3. Market Liquidity: A Data Check

_Days on Zillow_ looks like the natural liquidity metric: if it measured the time from listing to sale, a short distribution would signal a seller's market and a long tail would signal stale listings.

Before using it, check what it actually measures. If it were time to sale, most values should be weeks or a few months. We also add it to `date_posted` to see which date it counts up to.


In [16]:
print(df["days_on_zillow"].describe().round(0))
print(f"\nShare of properties with days_on_zillow > 365: {(df['days_on_zillow'] > 365).mean():.0%}")

implied_end = df["date_posted"] + pd.to_timedelta(df["days_on_zillow"], unit="D")
print("\nMonth of date_posted + days_on_zillow (top 6):")
print(implied_end.dt.strftime("%Y-%m").value_counts().head(6))

count    3736.0
mean      562.0
std       320.0
min         0.0
25%       279.0
50%       577.0
75%       861.0
max      1092.0
Name: days_on_zillow, dtype: float64

Share of properties with days_on_zillow > 365: 66%

Month of date_posted + days_on_zillow (top 6):
2026-02    1310
2026-01     802
2026-03     469
2025-12     318
2025-11     144
2025-10      92
Name: count, dtype: int64


The median is well over a year, and about two thirds of properties exceed 365 days. Adding `days_on_zillow` to `date_posted` lands mostly in the winter of 2025 to 2026, around when the data was collected. So `days_on_zillow` behaves like the number of days since the record was posted, counted up to data collection. It is not the time a property took to sell.

:::{caution} This dataset cannot measure time to sale

There is no sale date in the data, so we cannot say how quickly properties sell or which features are associated with faster sales. Answering those questions would need a sale date or a listing-event history (for example, the price-history file in the _Campustown Zillow Listings_ chapter).

:::


## 📈 4. Temporal Trends

Next, we look at how prices have changed month over month. We aggregate `date_posted` into year-month buckets and compute the median price for each bucket. The 438 properties with no `date_posted` cannot be placed in time, so they are left out here.

The first months of the data contain only a handful of properties each (one month has a single sale), and a median of one or two sales is mostly noise. We keep only months with at least 20 properties.


In [17]:
MIN_SALES_PER_MONTH = 20

dated = df.dropna(subset=["date_posted"])

df_temporal = (
    dated.groupby(dated["date_posted"].dt.strftime("%Y-%m"))
    .agg(
        median_price=("price", "median"),
        count=("property_id", "count"),
    )
    .reset_index()
    .rename(columns={"date_posted": "month_year"})
)

print(f"Months before filtering: {len(df_temporal)}")
df_temporal = df_temporal[df_temporal["count"] >= MIN_SALES_PER_MONTH]
print(f"Months with at least {MIN_SALES_PER_MONTH} properties: {len(df_temporal)}")

df_temporal.head(3)

Months before filtering: 47
Months with at least 20 properties: 39


,month_year,median_price,count
8,2023-01,379000.0,21
9,2023-02,220900.0,39
10,2023-03,199000.0,86


Plotting the resulting time series shows whether the local market is heating up, cooling down, or moving sideways. For an acquirer like Kingfisher, the slope of this line informs how aggressive their bid strategy should be.

In [18]:
fig = px.line(
    df_temporal,
    x="month_year",
    y="median_price",
    markers=True,
    title="Median Price Trends Over Time",
    labels={"median_price": "Median Price ($)", "month_year": "Date"},
    template="simple_white",
)
fig.show()

Monthly medians move around a lot, partly because the mix of homes changes from month to month. A yearly summary, including price per square foot (which adjusts for size), gives a steadier view. The first and last years are partial.

In [19]:
dated = df.dropna(subset=["date_posted"])

dated.groupby(dated["date_posted"].dt.year.rename("year")).agg(
    properties=("property_id", "count"),
    median_price=("price", "median"),
    median_price_per_sqft=("price_per_sqft", "median"),
).round(0)

,properties,median_price,median_price_per_sqft
year,,,
2022,31,265000.0,133.0
2023,1050,239900.0,141.0
2024,1070,255750.0,155.0
2025,1052,270500.0,161.0
2026,95,250000.0,166.0


## 🗺️ 5. Geographic Insights

Finally, we map every property and colour it by **price per square foot**. This shows _where in the metro_ you get the most house for your money.

Cool-coloured (low \$/sqft) clusters are candidates for a closer look, not automatic bargains: a low price per square foot can also reflect a home's age, condition, or location.


In [20]:
fig = px.scatter_map(
    df,
    lat="latitude",
    lon="longitude",
    color="price_per_sqft",
    hover_name="address_street",
    title="Geographic Value Map (Color=$/sqft)",
    color_continuous_scale=px.colors.sequential.ice,
    zoom=11,
    template="simple_white",
)
fig.show()

### School Premium

School quality is one of the strongest sub-market drivers in U.S. residential real estate. We approximate it here by computing the median sale price under each high-school catchment.

The `reso_high_school` field is typed in by listing agents, so the same school appears under several spellings (for example, `CENTENNIAL` and `Centennial High School`). We map the obvious variants of the three public high schools to one name and keep only catchments with at least 20 properties.


In [21]:
school_names = {
    "CENTENNIAL": "Centennial High School",
    "CENTENNIAL HIGH SCHOOL": "Centennial High School",
    "CENTRAL": "Central High School",
    "Central": "Central High School",
    "UHS": "Urbana High School",
    "Uhs": "Urbana High School",
    "URBANA HIGH SCHOOL": "Urbana High School",
}
df["high_school"] = df["reso_high_school"].replace(school_names)

df_school_prices = (
    df.groupby("high_school")
    .agg(properties=("property_id", "count"), price=("price", "median"))
    .query("properties >= 20")
    .sort_values("price", ascending=False)
    .reset_index()
)
df_school_prices

,high_school,properties,price
0,Central High School,1078,275000.0
1,Centennial High School,957,260000.0
2,Urbana High School,1009,235000.0
3,Champaign High School,195,220000.0
4,Unit 4 Of Choice,38,180000.0


In [22]:
fig = px.bar(
    df_school_prices,
    x="price",
    y="high_school",
    title="Median Sale Price by High-School Catchment (20+ properties)",
    labels={"price": "Median Price ($)", "high_school": ""},
    template="simple_white",
    height=400,
)
fig.update_yaxes(categoryorder="total ascending")
fig.show()

:::{important} Caveat: confounding
High-priced school catchments are _also_ often high-amenity neighborhoods, so the chart above bundles "good schools" with "good shopping", "low crime", "older trees", etc. Untangling these effects rigorously requires controlling for property characteristics, which is exactly what the regression and ML notebooks in other chapters teach.
:::


## 📌 Recap & Recommendation for Kingfisher

Pulling the threads together, our analysis suggests:

- **Pricing posture.** Median price per square foot rose in each full year of the data (from about \$141 in 2023 to about \$161 in 2025), and the median price rose with it. This looks like steady, modest appreciation, not a sharp run-up or a decline. Monthly medians are noisy, so judge the trend over years, not months.
- **Highest-value cities.** Savoy has the highest median price and the highest \$/sqft, with larger and newer homes. Champaign and Urbana have similar \$/sqft at a lower entry price, and Champaign offers by far the largest pool of properties, which gives the most room for value plays.
- **Feature focus.** Living area has the strongest link to price, followed by bathrooms and annual tax. Homes with a garage sell for much more at the median, but they are also larger and newer, so these gaps are not the payoff from adding a bathroom or a garage. Estimate feature effects with a model that holds size and location fixed before planning renovations around them.
- **Market liquidity.** This dataset cannot tell us how quickly properties sell: `days_on_zillow` counts days since posting, and there is no sale date. Ask the data vendor for sale dates or listing histories before making timing decisions.
- **Geography & schools.** The Central and Centennial high-school catchments have higher median prices than Urbana's. Those catchments also differ in home size and neighbourhood, so treat the gap as a starting point for a closer look, not as a school premium.

These findings are descriptive only. The next step in a real consulting engagement would be to fit a regression or gradient-boosted model to estimate fair value for each property and flag _under-priced_ properties to the acquisition team, using the same modelling pattern we work through in the _Australian Rental Market_ chapter.
